# Experiment 08 — Time homogeneity

**Paper reference.** §2.2 / Appendix B.1 of v15 (the prompt hides T to suppress deadline effects).

**Goal.** Fit the chosen linear model on three time windows (early t=3–5, middle 6–8, late 9–12) for each (scheme, policy) and compare coefficients. Drift across windows would indicate an unmodelled deadline effect that violates the time-homogeneous assumption in Definition 1.

In [ ]:
import os, sys, warnings
warnings.simplefilter('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
import negolin_v15 as nv
nv.setup_matplotlib()

DATA_DIR = '/home/claude/data/merged_results'    # <-- edit if needed
OUT_DIR  = './out'
os.makedirs(OUT_DIR, exist_ok=True)

raw = nv.load_run_files(DATA_DIR)
_, turns = nv.preprocess_turns(raw)
POLICIES = list(nv.POLICIES)
SCHEMES  = list(nv.SCHEMES)
print(f'rows={len(turns):,}  policies={POLICIES}  schemes={SCHEMES}')

In [ ]:
# Load the Stage-2 selection produced by 02_main_sweep.ipynb
sel_path = Path(OUT_DIR) / 'selected_linear.csv'
if sel_path.exists():
    SELECTED = pd.read_csv(sel_path)
    print('Loaded selected_linear.csv:'); display(SELECTED.head())
else:
    # Sensible fallback: Feature II + Ridge per (scheme, policy)
    rows = [{'scheme': s, 'policy': p, 'chosen_family': 'II', 'chosen_model': 'Ridge'}
            for s in SCHEMES for p in POLICIES]
    SELECTED = pd.DataFrame(rows)
    print('selected_linear.csv not found -> using fallback (II/Ridge)')

def picked(scheme, policy):
    r = SELECTED.set_index(['scheme','policy']).loc[(scheme, policy)]
    fam = r['chosen_family']
    if fam == 'Base': fam = 'II'    # Base isn't a feature spec -- fall back
    return fam, r['chosen_model']

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline

WINDOWS = {'early_3_5': (3, 5), 'middle_6_8': (6, 8), 'late_9_12': (9, 12)}

def fit_window(scheme, policy, lo, hi):
    fam, _ = picked(scheme, policy)
    cols = nv.FEATURE_SPECS[fam]['cols']
    sub = turns.query('scheme == @scheme and policy == @policy and round >= @lo and round <= @hi')
    meta, X, y, _ = nv.build_design_matrix(sub, fam, round_min=lo, round_max=hi)
    (_, _, _), (m_n, X_n, y_n) = nv.partition_persist(meta, X, y)
    if len(y_n) < 30:
        return None, None, fam
    est = Pipeline([('s', StandardScaler()), ('m', Ridge(alpha=1.0))]).fit(X_n, y_n)
    return dict(zip(cols, est.named_steps['m'].coef_)), len(y_n), fam

rows = []
for sch in SCHEMES:
    for pol in POLICIES:
        for w, (lo, hi) in WINDOWS.items():
            coefs, n, fam = fit_window(sch, pol, lo, hi)
            if coefs is None: continue
            for k, v in coefs.items():
                rows.append({'scheme': sch, 'policy': pol, 'window': w, 'family': fam,
                             'feat': k, 'coef': v, 'n': n})
time_df = pd.DataFrame(rows)
time_df.to_csv(os.path.join(OUT_DIR, 'exp08_time_coefs.csv'), index=False)
time_df.head(8).round(3)

## Coefficient drift across windows (scheme A as illustrative)

In [ ]:
piv = (time_df.query('scheme == "A"')
       .pivot_table(index=['policy','feat'], columns='window', values='coef'))
piv['drift_late_minus_early'] = piv.get('late_9_12', np.nan) - piv.get('early_3_5', np.nan)
piv.round(3).head(20)

In [ ]:
# Median absolute drift per (scheme, policy)
drift = (time_df.pivot_table(index=['scheme','policy','feat'], columns='window', values='coef')
         .assign(drift=lambda d: (d['late_9_12'] - d['early_3_5']).abs())
         .reset_index()
         .groupby(['scheme','policy'])['drift'].median()
         .unstack().round(3))
drift

Cells with median |drift| < ~0.10 standardized units are time-homogeneous in practice. Larger numbers signal that the chosen linear model needs a time-varying coefficient or a time feature; this is the natural launchpad for §3.3 (Adaptive Policy).